# Compare Lightweight Models (75 vs 100 vs 125 vs 150 Features)
This notebook systematically loads the exact lightweight models generated by the `train_lightgbm_reduced.ipynb` script. 

It safely streams the 10% test validation slice without overloading RAM, makes predictions using each model, and overlays their results (AUC, Accuracy, Precision, F1) to identify exactly how many features provide the optimal price-to-performance ratio.

In [ ]:
import os
import sys
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, roc_curve

# Display plots inline
%matplotlib inline

# Ensure the paths to the virtual environment(s) are added BEFORE importing packages
base_dir = os.getcwd()
for venv_folder in ["venv", ".venv"]:
    site_packages = os.path.join(base_dir, venv_folder, "Lib", "site-packages")
    if os.path.exists(site_packages) and site_packages not in sys.path:
        sys.path.insert(0, site_packages)
        break

global_venv_site_packages = r"Z:\ai project\.venv\Lib\site-packages"
if os.path.exists(global_venv_site_packages) and global_venv_site_packages not in sys.path:
    sys.path.insert(0, global_venv_site_packages)

try:
    from thrember.features import PEFeatureExtractor
    print("Features extractor imported successfully.")
except ImportError:
    print("Warning: 'thrember' not found. Will use default feature dimension (2381).")
    class PEFeatureExtractor:
        dim = 2381

### 1. Configure Testing Environment
Calculate the 10% test split from the original .dat files.

In [ ]:
DATASET_DIR = r"Z:\ember2024_train_data"
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

extractor = PEFeatureExtractor()
ndim = extractor.dim

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)

# Calculate Split (Last 10% is test)
train_nrows = int(nrows * 0.9)
val_nrows = nrows - train_nrows
val_start_idx = train_nrows

print(f"Global dataset size: {nrows} rows.")
print(f"Evaluation target: exactly the last 10% ({val_nrows} rows) starting at index {val_start_idx}.")

# Open memory mapped pointers
X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

### 2. Predict & Evaluate Across All Configurations

In [ ]:
feature_counts = [75, 100, 125, 150]
results = []
roc_data = {}
BATCH_SIZE = 100000

for num_fe in feature_counts:
    print(f"\n{'='*60}\n🔬 EVALUATING MODEL WITH {num_fe} FEATURES\n{'='*60}")
    
    model_path = os.path.join(DATASET_DIR, f"ember_model_reduced_{num_fe}.txt")
    indices_path = os.path.join(DATASET_DIR, f"ember_reduced_{num_fe}_indices.npy")
    
    if not os.path.exists(model_path) or not os.path.exists(indices_path):
        print(f"⚠️ Missing model or indices for {num_fe} features. Skipping.")
        continue
        
    # Load Model and its specific Feature Index rules
    model = lgb.Booster(model_file=model_path)
    current_indices = np.load(indices_path)
    
    y_true_all = []
    y_pred_prob_all = []

    print("Streaming and predicting testing data chunks...")
    for batch_start in range(val_start_idx, nrows, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, nrows)
        
        # Stream batch from disk and instantly slice to only the required features
        X_batch = np.array(X_memmap[batch_start:batch_end])[:, current_indices]
        y_batch = np.array(y_memmap[batch_start:batch_end])
        
        # Drop unlabeled data
        valid_mask = y_batch != -1
        if not np.any(valid_mask):
            continue
            
        X_batch = X_batch[valid_mask]
        y_batch = y_batch[valid_mask]
        
        preds = model.predict(X_batch)
        
        y_true_all.extend(y_batch)
        y_pred_prob_all.extend(preds)
        
    y_true = np.array(y_true_all)
    y_pred_probs = np.array(y_pred_prob_all)
    y_pred = (y_pred_probs > 0.5).astype(int)
    
    # Calculate Metrics
    auc = roc_auc_score(y_true, y_pred_probs)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    print(f"Test AUC:       {auc:.4f}")
    print(f"Test Accuracy:  {acc:.4f}")
    print(f"Test F1 Score:  {f1:.4f}")
    
    results.append({
        "Num Features": num_fe,
        "AUC": round(auc, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4)
    })
    
    # Cache the ROC curve vectors so we can overlay them at the end
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
    roc_data[num_fe] = (fpr, tpr, auc)

### 3. Comparison Reports & Visualizations

In [ ]:
df_results = pd.DataFrame(results)

print("\n🚀 FINAL COMPARISON RESULTS 🚀\n")
display(df_results)

# Save to CSV
report_path = os.path.join(DATASET_DIR, "feature_reduction_evaluation_metrics.csv")
df_results.to_csv(report_path, index=False)
print(f"📄 Full comparison metrics saved to: {report_path}")

# Plot AUC Comparison
sns.set_theme(style="whitegrid")
plt.figure(figsize=(9, 5))
plot = sns.barplot(data=df_results, x='Num Features', y='AUC', palette='viridis')
plt.title('AUC Score vs Feature Count')
plt.ylabel('AUC Score')
plt.xlabel('Number of Top Features Kept')

# Zoom vertically if AUC > 0.9 for visual contrast
min_auc = df_results['AUC'].min()
if min_auc > 0.90:
    plt.ylim(0.90, 1.0)
elif min_auc > 0.80:
    plt.ylim(0.80, 1.0)

# Annotation Text
for index, value in enumerate(df_results['AUC']):
    plot.text(index, value + 0.002, f"{value:.4f}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Plot Overlayed ROC Curves
plt.figure(figsize=(10, 8))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, num_fe in enumerate(roc_data):
    fpr, tpr, auc = roc_data[num_fe]
    plt.plot(fpr, tpr, label=f'{num_fe} Features (AUC = {auc:.4f})', linewidth=2.5, alpha=0.85, color=colors[i % len(colors)])

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, alpha=0.5, label='Random Chance')
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Performance Curve Overlay Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', frameon=True, fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()